In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip uninstall -y numpy
!pip install -q numpy==1.26.4
!pip install -q transformers==4.43.4 accelerate==0.33.0
!pip install -q -U bitsandbytes
!pip install -q torch scikit-learn networkx
!pip install -q sinling sentencepiece

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is inc

In [3]:
import os
import json
import torch
import torch.nn as nn
import numpy as np
import networkx as nx
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig
from sinling import SinhalaTokenizer, POSTagger
from sklearn.cluster import AgglomerativeClustering
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


print(' Libraries imported')

 Libraries imported


In [ ]:
BASE_PATH = 'YOUR_PATH'
TEST_PATH = f'{BASE_PATH}/YOUR_PATH'
MODEL_PATH = f'{BASE_PATH}/YOUR_PATH'
OUTPUT_PATH = f'{BASE_PATH}/YOUR_PATH'
SUMMARY_OUTPUT = f'{OUTPUT_PATH}/YOUR_PATH'

os.makedirs(SUMMARY_OUTPUT, exist_ok=True)

assert os.path.exists(TEST_PATH), f'Test path not found: {TEST_PATH}'
assert os.path.exists(MODEL_PATH), f'Model path not found: {MODEL_PATH}'

print(' Paths configured:')
print(f'   Test docs: {TEST_PATH}')
print(f'   Model: {MODEL_PATH}')
print(f'   Output: {SUMMARY_OUTPUT}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

 Paths configured:
   Test docs: /content/drive/MyDrive/SinBrief_Implementation/Dataset/Test
   Model: /content/drive/MyDrive/SinBrief_Implementation/Models
   Output: /content/drive/MyDrive/SinBrief_Implementation/Outputs/Falcon_Summaries
Using device: cuda


In [5]:
# Sinhala tools
tokenizer_sin = SinhalaTokenizer()
pos_tagger = POSTagger()

# Legal keywords for weighted graphs
LEGAL_KEYWORDS = [
    'අධිකරණය',
    'නීතිය',
    'විනිශ්චය',
    'දඬුවම',
    'චෝදනා',
    'සාක්ෂි',
    'තීන්දුව',
    'අභියාචනය',
    'නඩුව',
    'වරද',
    'දෝෂ',
    'බලය',
    'කොන්දේසි',
    'ප්‍රතිපාදන',
    'ව්‍යවස්ථා',
    'පනත',
    'පැමිණිලිකරු',
    'කොන්ත්‍රාත්තුව',
    'පැමිණිල්ල'
]

print(f" Loaded {len(LEGAL_KEYWORDS)} legal keywords")

 Loaded 19 legal keywords


In [6]:
from huggingface_hub import notebook_login
notebook_login()

In [7]:
FALCON_MODEL_NAME = 'tiiuae/falcon-7b'

print('Loading Falcon 7B tokenizer...')
tokenizer_falcon = AutoTokenizer.from_pretrained(FALCON_MODEL_NAME, trust_remote_code=True)
tokenizer_falcon.pad_token = tokenizer_falcon.eos_token

# Same 4-bit config as training
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

from transformers import AutoModelForCausalLM

print('Loading Falcon 7B with AutoModelForCausalLM...')
model_falcon = AutoModelForCausalLM.from_pretrained(
    FALCON_MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    use_cache=False
)
model_falcon.eval()

model_device = next(model_falcon.parameters()).device
FALCON_HIDDEN_SIZE = model_falcon.config.hidden_size
print(f' Falcon 7B loaded. Device: {model_device}, Hidden size: {FALCON_HIDDEN_SIZE}')

Loading Falcon 7B tokenizer...


Loading Falcon 7B with AutoModelForCausalLM...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

 Falcon 7B loaded. Device: cuda:0, Hidden size: 4544


In [9]:
test_inputs = tokenizer_falcon(
    "test sentence",
    return_tensors='pt',
    max_length=512,
    truncation=True,
    padding=True
).to(model_device)

with torch.no_grad():
    out = model_falcon.transformer(
        input_ids=test_inputs['input_ids'],
        attention_mask=test_inputs['attention_mask']
    )
print(type(out))
print(dir(out))

<class 'transformers.modeling_outputs.BaseModelOutputWithPastAndCrossAttentions'>
['__annotations__', '__class__', '__class_getitem__', '__contains__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__ior__', '__iter__', '__le__', '__len__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__or__', '__post_init__', '__reduce__', '__reduce_ex__', '__repr__', '__reversed__', '__ror__', '__setattr__', '__setitem__', '__sizeof__', '__str__', '__subclasshook__', 'attentions', 'clear', 'copy', 'cross_attentions', 'fromkeys', 'get', 'hidden_states', 'items', 'keys', 'last_hidden_state', 'move_to_end', 'past_key_values', 'pop', 'popitem', 'setdefault', 'to_tuple', 'update', 'values']


In [10]:
def get_sentence_embedding(sentence):
    inputs = tokenizer_falcon(
        sentence,
        return_tensors='pt',
        max_length=512,
        truncation=True,
        padding=True
    ).to(model_device)

    with torch.no_grad():
        outputs = model_falcon.transformer(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )

    # Use last_hidden_state directly
    last_hidden_state = outputs.last_hidden_state

    attention_mask = inputs['attention_mask']
    last_token_idx = attention_mask.sum(dim=1) - 1
    batch_idx = torch.arange(last_hidden_state.size(0), device=model_device)
    embedding = last_hidden_state[batch_idx, last_token_idx, :]

    return embedding.squeeze().cpu().float().numpy()

print(' Sentence embedding function defined')

 Sentence embedding function defined


In [11]:
def cluster_sentences(sentences, num_clusters=3):
    """
    Cluster sentences by semantic similarity
    """
    if len(sentences) <= num_clusters:
        return [[sent] for sent in sentences]

    print(f'   Generating embeddings for {len(sentences)} sentences...')
    embeddings = np.array([get_sentence_embedding(sent) for sent in sentences])

    clustering = AgglomerativeClustering(
        n_clusters=min(num_clusters, len(sentences)),
        metric='cosine',
        linkage='average'
    )
    labels = clustering.fit_predict(embeddings)

    clusters = {}
    for idx, label in enumerate(labels):
        if label not in clusters:
            clusters[label] = []
        clusters[label].append(sentences[idx])

    return list(clusters.values())

print(' Clustering function defined')

 Clustering function defined


In [12]:
KEEP_POS = {'NOUN', 'VERB', 'AUX', 'PROPN', 'ADJ'}

def build_word_graph_weighted(sentences_in_cluster, legal_keywords):
    """
    Build word graph with LEGAL KEYWORD-AWARE EDGE WEIGHTS
    FIXED: More robust - doesn't rely solely on POS tagging
    """
    G = nx.DiGraph()
    G.add_node('START')
    G.add_node('END')

    total_words_added = 0

    for sentence in sentences_in_cluster:
        tokenized = tokenizer_sin.tokenize(sentence)
        if len(tokenized) == 0:
            continue

        words_with_pos = []
        try:
            pos_tags = pos_tagger.predict([tokenized])[0]
            words_with_pos = [(w, p) for w, p in zip(tokenized, pos_tags)
                             if p in KEEP_POS and len(w) > 1]
        except:
            pass

        if len(words_with_pos) < 3:
            words_with_pos = []
            for w in tokenized:
                if len(w) > 2:
                    words_with_pos.append((w, 'NOUN'))

        words_with_pos = words_with_pos[:30]

        if len(words_with_pos) == 0:
            continue

        first_word = words_with_pos[0]
        weight = 2.0 if any(kw in first_word[0] for kw in legal_keywords) else 1.0
        if G.has_edge('START', first_word):
            G['START'][first_word]['weight'] += weight
        else:
            G.add_edge('START', first_word, weight=weight)
        total_words_added += 1

        for i in range(len(words_with_pos)-1):
            current = words_with_pos[i]
            next_word = words_with_pos[i+1]
            has_keyword = any(kw in current[0] or kw in next_word[0] for kw in legal_keywords)
            weight = 2.0 if has_keyword else 1.0
            if G.has_edge(current, next_word):
                G[current][next_word]['weight'] += weight
            else:
                G.add_edge(current, next_word, weight=weight)
            total_words_added += 1

        last_word = words_with_pos[-1]
        weight = 2.0 if any(kw in last_word[0] for kw in legal_keywords) else 1.0
        if G.has_edge(last_word, 'END'):
            G[last_word]['END']['weight'] += weight
        else:
            G.add_edge(last_word, 'END', weight=weight)

    print(f'       Built graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
    return G

print(' Word graph building function defined')

 Word graph building function defined


In [13]:
import random

def generate_fused_sentences(G, num_paths=5, max_length=25, min_length=8):
    """
    Generate high-quality abstractive candidates
    IMPROVED: Better coherence, length control, deduplication
    """
    if G.number_of_nodes() <= 2:
        return []

    candidates = []
    seen_word_sets = set()

    for attempt in range(num_paths * 5):
        try:
            current = 'START'
            words = []

            for step in range(max_length + 5):
                neighbors = list(G.successors(current))

                if len(words) < min_length and 'END' in neighbors:
                    neighbors.remove('END')

                if len(neighbors) == 0:
                    break

                if len(words) >= min_length and 'END' in neighbors:
                    if random.random() < 0.3:
                        break

                weights = []
                valid_neighbors = []
                for n in neighbors:
                    if n != 'END':
                        weights.append(G[current][n].get('weight', 1.0))
                        valid_neighbors.append(n)

                if len(valid_neighbors) == 0:
                    break

                next_node = random.choices(valid_neighbors, weights=weights)[0]

                if isinstance(next_node, tuple):
                    word = next_node[0]
                    if len(word) > 1:
                        words.append(word)

                current = next_node

                if len(words) >= max_length:
                    break

            if len(words) < min_length or len(words) > max_length:
                continue

            word_set = frozenset(words)
            if word_set in seen_word_sets:
                continue
            seen_word_sets.add(word_set)

            fused = ' '.join(words)

            if len(fused) < 20:
                continue

            unique_words = len(set(words))
            if unique_words < len(words) * 0.6:
                continue

            candidates.append(fused)

            if len(candidates) >= num_paths:
                break

        except Exception:
            continue

    if len(candidates) < num_paths:
        word_weights = {}
        for u, v, data in G.edges(data=True):
            for node in [u, v]:
                if isinstance(node, tuple) and node not in ['START', 'END']:
                    word = node[0]
                    if len(word) > 1:
                        weight = data.get('weight', 1.0)
                        word_weights[word] = word_weights.get(word, 0) + weight

        if len(word_weights) >= min_length:
            sorted_words = sorted(word_weights.items(), key=lambda x: x[1], reverse=True)
            attempts = 0
            while len(candidates) < num_paths and attempts < 10:
                attempts += 1
                n_words = random.randint(min_length, min(max_length, len(sorted_words)))
                if attempts == 1:
                    selected = [w for w, _ in sorted_words[:n_words]]
                else:
                    top_k = len(sorted_words) // 2
                    top_words = [w for w, _ in sorted_words[:top_k]]
                    selected = random.sample(top_words, min(n_words, len(top_words)))
                word_set = frozenset(selected)
                if word_set in seen_word_sets:
                    continue
                seen_word_sets.add(word_set)
                fused = ' '.join(selected)
                if len(fused) >= 20:
                    candidates.append(fused)

    return candidates[:num_paths]


def filter_low_quality_candidates(candidates):
    """
    Filter out obviously bad candidates
    """
    from collections import Counter
    filtered = []
    for candidate in candidates:
        if len(candidate) < 20:
            continue
        words = candidate.split()
        unique_ratio = len(set(words)) / len(words)
        if unique_ratio < 0.6:
            continue
        word_counts = Counter(words)
        if max(word_counts.values()) >= 3:
            continue
        filtered.append(candidate)
    return filtered if len(filtered) > 0 else candidates

print(' Candidate generation & filtering functions defined')

 Candidate generation & filtering functions defined


In [14]:
class LegalSentenceScorer_Falcon(nn.Module):
    def __init__(self, base_model, hidden_size=4544):
        super().__init__()
        self.base_model = base_model
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask):
      outputs = self.base_model.transformer(
          input_ids=input_ids,
          attention_mask=attention_mask
      )
      last_hidden_state = outputs.last_hidden_state
      sequence_lengths = attention_mask.sum(dim=1) - 1
      batch_size = input_ids.shape[0]
      last_token_embedding = last_hidden_state[
          torch.arange(batch_size, device=last_hidden_state.device),
          sequence_lengths
      ]
      last_token_embedding = last_token_embedding.to(torch.float32)
      score = self.classifier(last_token_embedding)
      return score.squeeze()


# Build scorer using already-loaded model_falcon as backbone
print('Building scorer model...')
scorer_model = LegalSentenceScorer_Falcon(model_falcon, hidden_size=FALCON_HIDDEN_SIZE)
scorer_model.classifier = scorer_model.classifier.to(model_device)

# Load only classifier weights — matches how training saved the checkpoint
print('Loading trained classifier weights...')
checkpoint = torch.load(f'{MODEL_PATH}/falcon_sentence_scorer.pt', map_location=model_device)
scorer_model.classifier.load_state_dict(checkpoint['classifier_state_dict'])
scorer_model.eval()

print(f' Falcon scorer loaded (Val Accuracy: {checkpoint["best_val_accuracy"]:.4f})')

Building scorer model...
Loading trained classifier weights...
 Falcon scorer loaded (Val Accuracy: 0.8483)


In [15]:
def generate_summary_with_scoring(document, num_clusters=3):
    """
    Generate abstractive summary with improved quality.
    Longer output, multiple sentences per cluster, complete endings.
    """
    import re
    split_pattern = re.compile(r'[.!?;]+')
    sentences = [s.strip() for s in split_pattern.split(document)
                if len(s.strip()) > 20]

    if len(sentences) < 3:
        return '. '.join(sentences) + '.'

    print(f'   Clustering {len(sentences)} sentences...')
    clusters = cluster_sentences(sentences, num_clusters=min(3, max(2, len(sentences)//5)))
    print(f'   Created {len(clusters)} clusters')

    summary_sentences = []

    for cluster_idx, cluster in enumerate(clusters, 1):
        if len(cluster) == 0:
            continue

        print(f'   Processing cluster {cluster_idx}/{len(clusters)} ({len(cluster)} sentences)...')

        G = build_word_graph_weighted(cluster, LEGAL_KEYWORDS)

        if G.number_of_nodes() <= 10:
            print(f'       Graph has only {G.number_of_nodes()} nodes, skipping')
            continue

        candidates = generate_fused_sentences(G, num_paths=8, max_length=25, min_length=8)

        if len(candidates) == 0:
            print(f'      No candidates generated, skipping cluster')
            continue

        candidates = filter_low_quality_candidates(candidates)

        if len(candidates) == 0:
            print(f'       All candidates filtered out, skipping cluster')
            continue

        print(f'     Generated {len(candidates)} abstractive candidates, scoring...')

        candidate_scores = []

        for cand_idx, candidate in enumerate(candidates):
            encoding = tokenizer_falcon(
                candidate,
                add_special_tokens=True,
                max_length=128,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )

            with torch.no_grad():
                score = scorer_model(
                    encoding['input_ids'].to(model_device),
                    encoding['attention_mask'].to(model_device)
                ).item()

            candidate_scores.append((candidate, score))
            print(f'       Candidate {cand_idx+1}: {score:.4f} | {candidate[:60]}...')

        candidate_scores.sort(key=lambda x: x[1], reverse=True)

        top_k = min(2, len(candidate_scores))
        for i in range(top_k):
            candidate, score = candidate_scores[i]
            print(f'     Selected #{i+1} (score: {score:.4f})')
            summary_sentences.append(candidate)

    if len(summary_sentences) == 0:
        return '. '.join(sentences[:3]) + '.'

    summary = '. '.join(summary_sentences)

    target_min = 300
    target_max = 500

    if len(summary) < target_min:
        if not summary.endswith('.'):
            summary += '.'
    elif len(summary) <= target_max:
        if not summary.endswith('.'):
            summary += '.'
    else:
        sentences_in_summary = [s.strip() for s in summary.split('.') if s.strip()]
        final_summary = ''
        for sent in sentences_in_summary:
            test_summary = final_summary + sent + '. '
            if len(test_summary) > target_max:
                if len(final_summary) > target_min:
                    break
                else:
                    final_summary = test_summary
                    break
            else:
                final_summary = test_summary
        summary = final_summary.strip()
        if not summary.endswith('.'):
            last_space = summary.rfind(' ')
            if last_space > target_min:
                summary = summary[:last_space] + '.'
            else:
                summary += '.'

    if not summary.endswith('.'):
        summary += '.'

    return summary

print(' Summary generation function defined')

 Summary generation function defined


In [16]:
test = get_sentence_embedding("test sentence")
print(test.shape)
print("SUCCESS")

(4544,)
SUCCESS


In [17]:
test_files = [f for f in os.listdir(TEST_PATH) if f.endswith('.txt')]
print(f' Found {len(test_files)} test documents')

print('\n' + '='*60)
print('GENERATING SUMMARIES (FALCON 7B SCORER)')
print('='*60 + '\n')

successful = 0
failed = 0
failed_files = []

for i, filename in enumerate(test_files, 1):
    print(f'\n[{i}/{len(test_files)}] Processing: {filename}')
    print('-' * 60)

    try:
        filepath = os.path.join(TEST_PATH, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            document = f.read().strip()

        if len(document) < 100:
            print('     Document too short, skipping')
            failed += 1
            failed_files.append(filename)
            continue

        summary = generate_summary_with_scoring(document, num_clusters=3)

        base_name = os.path.splitext(filename)[0]
        output_file = os.path.join(SUMMARY_OUTPUT, f'{base_name}_summary.txt')

        with open(output_file, 'w', encoding='utf-8') as out:
            out.write(summary)

        print(f'\n    Summary generated ({len(summary)} chars)')
        print(f'    Preview: {summary[:150]}...')

        successful += 1

    except Exception as e:
        print(f'    Error: {e}')
        failed += 1
        failed_files.append(filename)

print('\n' + '='*60)
print('SUMMARY GENERATION COMPLETE')
print('='*60)
print(f'\n Successfully generated: {successful} summaries')
print(f' Failed: {failed} documents')

if failed_files:
    print(f'\n  Failed files:')
    for f in failed_files[:10]:
        print(f'   - {f}')

print(f'\n Summaries saved to: {SUMMARY_OUTPUT}')

Streaming output truncated to the last 5000 lines.
     Selected #1 (score: 0.5555)
     Selected #2 (score: 0.3759)
   Processing cluster 2/3 (13 sentences)...
       Built graph: 230 nodes, 351 edges
     Generated 7 abstractive candidates, scoring...
       Candidate 1: 0.4887 | මෙම පනතේ රීතීන්ට අනුකූලව තෝරා පත්කර ගනු ලබන චන්දිම වීරක්කොඩි...
       Candidate 2: 0.2780 | සංස්ථාවේ කාර්ය සඳහා අවශ්‍ය වන්නා චංචල නිශ්චල දේපලක් අත්කර ගැ...
       Candidate 3: 0.4878 | සංස්ථාව ස්වේච්ඡා සමාජ සේවා සංවිධාන ලියාපදිංචි කිරීම සඳහා සංස...
       Candidate 4: 0.4411 | සභාපති ලේකම් භාණ්ඩාගාරික සංස්ථාවේ කටයුතු මෙම පනත චන්දිම වීරක...
       Candidate 5: 0.2689 | යම් පළාත් සභාවක අමාත්‍යාංශයක් දෙපාර්තමේන්තුවක් විසින් නිකුත්...
       Candidate 6: 0.3917 | සංස්ථාව වෙත සතු යම් ආයතනයක සාමාජිකයකු වන්නා චංචල නිශ්චල දේපළ...
       Candidate 7: 0.6371 | සංස්ථාව විසින් සමගාමීව කරගෙන යනු ලබන චන්දිම වීරක්කොඩි පදනම ස...
     Selected #1 (score: 0.6371)
     Selected #2 (score: 0.4887)
   Processing cluster 3/3 (18

In [18]:
stats = {
    'total_test_documents': len(test_files),
    'successful_summaries': successful,
    'failed_documents': failed,
    'success_rate': successful / len(test_files) * 100 if len(test_files) > 0 else 0,
    'output_directory': SUMMARY_OUTPUT,
    'scorer_model': 'Falcon 7B',
    'modifications': [
        'Legal keyword-weighted graph construction',
        'Multi-path candidate generation with Falcon 7B neural scoring'
    ],
    'parameters': {
        'num_clusters': 3,
        'num_paths_per_cluster': 5,
        'max_sentence_length': 20,
        'summary_max_length': 300
    }
}

stats_file = f'{OUTPUT_PATH}/summary_generation_stats_falcon.json'
with open(stats_file, 'w', encoding='utf-8') as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print(f' Statistics saved to {stats_file}')

 Statistics saved to /content/drive/MyDrive/SinBrief_Implementation/Outputs/summary_generation_stats_falcon.json
